In [8]:
import pandas as pd
genetic_data= pd.read_csv(r"C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\GeneMarkers-95 varieties 1.csv")
genetic_data.head()

,taglo_id,ATLANTIC_124776,ESCORT_159228,ARGOS_120931,ALTURAS_329540,ELMUNDO_835520,NADINE_241950,VIOLETQUEEN_3345402,DEODARA_513721,MEMPHIS_2279529,...,DIAMANT_153502,INNOVATOR_234757,TRIPLE7_3347283,SPUNTA_283648,ANTI_118190,CARRERA_142554,FORTUS_3279866,SMART_1883446,SALINERO_253609,FABULA_171173
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,17,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
genetic_data.info
genetic_data.head(10)

,taglo_id,ATLANTIC,ESCORT,ARGOS,ALTURAS,ELMUNDO,NADINE,VIOLETQUEEN,DEODARA,MEMPHIS,...,DIAMANT,INNOVATOR,TRIPLE7,SPUNTA,ANTI,CARRERA,FORTUS,SMART,SALINERO,FABULA
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,17,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,22,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,39,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,48,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,49,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,54,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [4]:
genetic_data.isnull().sum()

taglo_id           0
ATLANTIC_124776    0
ESCORT_159228      0
ARGOS_120931       0
ALTURAS_329540     0
                  ..
CARRERA_142554     0
FORTUS_3279866     0
SMART_1883446      0
SALINERO_253609    0
FABULA_171173      0
Length: 95, dtype: int64

In [6]:
aroma_mapped_clean= pd.read_csv(r"C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\compound_analysis\aroma_mapped_separate_columns.csv")
aroma_mapped_clean

,SampleNumber,Variety,FileName,Aroma Compound,Retention Time (min)
0,1,ADORA,sample_1.csv,"Butanal, 3-methyl-",2.893083
1,1,ADORA,sample_1.csv,"Cyclotrisiloxane, hexamethyl-",5.902667
2,1,ADORA,sample_1.csv,Benzaldehyde,9.337945
3,1,ADORA,sample_1.csv,"Cyclotrisiloxane, hexamethyl-",10.117983
4,1,ADORA,sample_1.csv,Octanal,10.220883
...,...,...,...,...,...
792,94,VIOLET QUEEN,sample_94.csv,"Cyclotrisiloxane, hexamethyl-",5.894919
793,94,VIOLET QUEEN,sample_94.csv,Benzaldehyde,9.330250
794,94,VIOLET QUEEN,sample_94.csv,"Cyclotrisiloxane, hexamethyl-",10.114800
795,94,VIOLET QUEEN,sample_94.csv,Nonanal,12.105100


In [29]:

# Extract just the variety names (before the underscore)
genetic_varieties = [col.split('_')[0].upper() for col in genetic_data.columns if col != 'taglo_id']

# Compare with aroma dataset
aroma_varieties = set(aroma_mapped_clean['Variety'].str.upper())
genetic_varieties_set = set(genetic_varieties)



genetic_data.columns = ['taglo_id'] + genetic_varieties 

def normalize_variety(name):
    return name.split('_')[0].replace(' ', '').replace('-', '').upper()

# Normalize both datasets
aroma_mapped_clean['norm'] = aroma_mapped_clean['Variety'].apply(normalize_variety)
genetic_data.columns = [normalize_variety(c) if c != 'taglo_id' else c for c in genetic_data.columns]

# Now match
matched = set(aroma_mapped_clean['norm']).intersection(genetic_data.columns)
missing = set(aroma_mapped_clean['norm']) - matched
print("Missing varieties after normalization:", missing)

# Normalize aroma dataset varieties
aroma_mapped_clean['Variety_norm'] = aroma_mapped_clean['Variety'].apply(normalize_name)

# Normalize genetic dataset columns
genetic_renamed = {col: normalize_name(col.split('_')[0]) for col in genetic_data.columns if col != 'taglo_id'}

# Replace columns with normalized names
genetic_data.columns = ['taglo_id'] + list(genetic_renamed.values())


Missing varieties after normalization: {'ALVERSTONER.'}


In [31]:
# Identify which varieties in aroma are in genetics
matching_varieties = set(aroma_mapped_clean['Variety_norm']).intersection(set(genetic_data.columns))
print(f"Matching varieties: {len(matching_varieties)}")


Matching varieties: 89


In [16]:
from difflib import get_close_matches

# Varieties in each dataset
aroma_varieties = set(aroma_mapped_clean['Variety_norm'])
genetic_varieties = set(genetic_data.columns[1:])  # skip taglo_id

# Find unmatched
unmatched = aroma_varieties - genetic_varieties
print("Unmatched varieties:", unmatched)

# Try fuzzy matching for each unmatched
for v in unmatched:
    close = get_close_matches(v, genetic_varieties, n=1, cutoff=0.6)
    print(f"{v} -> {close}")


Unmatched varieties: set()
